# Linear Models & Gradient Descent from Scratch

Implementing linear models from scratch is a near-universal MLE coding question because it tests vectorized math, gradient derivations, numerical hygiene, and optimizer design simultaneously. This note covers linear/logistic regression, regularization, and the optimizer zoo — all verified against scikit-learn.

## What Interviewers Test
- Can you derive and implement gradients from first principles (not just copy formulas)?
- Do you vectorize correctly across batch dimensions?
- Can you implement SGD, momentum, and Adam without looking them up?
- Do you know why feature standardization matters before GD?
- Can you diagnose learning-rate issues from a loss curve?
- Do you know the closed-form solution for linear regression and when to prefer it over GD?

## Linear Regression

### Closed-Form (Normal Equation)
$$\hat{\theta} = (X^T X)^{-1} X^T y$$

**When to use:** $n < 10^4$ features; no need for iterative training. $O(d^3)$ to invert, so impractical for large feature dimensions.

### Batch Gradient Descent
Loss: $L = \frac{1}{2n}\|X\theta - y\|^2$

Gradient: $\nabla_\theta L = \frac{1}{n} X^T(X\theta - y)$


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Synthetic data ---
n, d = 200, 5
X = np.random.randn(n, d)
true_w = np.array([1.5, -2.0, 0.5, 0.0, 3.0])
y = X @ true_w + 0.3 * np.random.randn(n)

# Add bias column
X_b = np.hstack([X, np.ones((n, 1))])  # (n, d+1)

# ---- Closed-form ----
def linear_regression_normal(X, y):
    """Normal equation: theta = (X^T X)^{-1} X^T y"""
    return np.linalg.lstsq(X, y, rcond=None)[0]  # lstsq is numerically safer than inv

theta_cf = linear_regression_normal(X_b, y)

# ---- Batch GD ----
def linear_regression_gd(X, y, lr=0.01, n_iter=500):
    n, d = X.shape
    theta = np.zeros(d)
    losses = []
    for _ in range(n_iter):
        preds = X @ theta
        residuals = preds - y
        loss = 0.5 * np.mean(residuals**2)
        grad = X.T @ residuals / n           # (d,)
        theta -= lr * grad
        losses.append(loss)
    return theta, losses

theta_gd, losses = linear_regression_gd(X_b, y, lr=0.05, n_iter=300)

# Verify against sklearn
sk = LinearRegression().fit(X, y)
theta_sk = np.append(sk.coef_, sk.intercept_)

print("=== Linear Regression Weight Comparison ===")
print(f"Normal eq: {theta_cf[:5].round(3)}")
print(f"Batch GD:  {theta_gd[:5].round(3)}")
print(f"sklearn:   {theta_sk[:5].round(3)}")
print(f"All close (CF vs sklearn): {np.allclose(theta_cf, theta_sk, atol=1e-3)}")
print(f"All close (GD vs sklearn): {np.allclose(theta_gd, theta_sk, atol=0.05)}")


## Logistic Regression

Loss (binary cross-entropy): $L = -\frac{1}{n}\sum_i [y_i \log \sigma(z_i) + (1-y_i)\log(1-\sigma(z_i))]$

Gradient: $\nabla_\theta L = \frac{1}{n} X^T(\sigma(X\theta) - y)$

> 💡 **Interview Tip:** The gradient has the same *form* as linear regression — $(\hat{y} - y)$ — but $\hat{y}$ is now $\sigma(X\theta)$. This is not a coincidence; it comes from the exponential family / GLM structure. Mentioning this impresses interviewers.


In [ ]:
def sigmoid(z):
    """Numerically stable sigmoid."""
    return np.where(z >= 0,
                    1 / (1 + np.exp(-z)),
                    np.exp(z) / (1 + np.exp(z)))

def logistic_regression_gd(X, y, lr=0.1, n_iter=500):
    n, d = X.shape
    theta = np.zeros(d)
    losses = []
    for _ in range(n_iter):
        z = X @ theta
        y_hat = sigmoid(z)
        # Clip to avoid log(0)
        y_hat_clipped = np.clip(y_hat, 1e-9, 1 - 1e-9)
        loss = -np.mean(y * np.log(y_hat_clipped) + (1 - y) * np.log(1 - y_hat_clipped))
        grad = X.T @ (y_hat - y) / n
        theta -= lr * grad
        losses.append(loss)
    return theta, losses

# Binary classification data
X_cls = np.random.randn(300, 4)
true_w_cls = np.array([1.0, -1.5, 0.5, 2.0])
y_cls = (sigmoid(X_cls @ true_w_cls) > 0.5).astype(float)
X_cls_b = np.hstack([X_cls, np.ones((300, 1))])

theta_lr, losses_lr = logistic_regression_gd(X_cls_b, y_cls, lr=0.5, n_iter=300)

# Verify against sklearn
sk_lr = LogisticRegression(C=1e6, max_iter=500).fit(X_cls, y_cls)
print("=== Logistic Regression ===")
print(f"From scratch: {theta_lr[:4].round(3)}")
sk_coef = np.append(sk_lr.coef_[0], sk_lr.intercept_[0])
print(f"sklearn:      {sk_coef[:4].round(3)}")

# Accuracy
preds_scratch = (sigmoid(X_cls_b @ theta_lr) > 0.5).astype(int)
acc_scratch = np.mean(preds_scratch == y_cls)
acc_sk = np.mean(sk_lr.predict(X_cls) == y_cls)
print(f"Accuracy scratch: {acc_scratch:.3f}, sklearn: {acc_sk:.3f}")


## Ridge & Lasso Regularization

**Ridge (L2):** $L = \text{MSE} + \lambda \|\theta\|^2$  → gradient adds $2\lambda\theta$

**Lasso (L1):** $L = \text{MSE} + \lambda \|\theta\|_1$  → non-differentiable at 0; use coordinate descent

### Coordinate Descent for Lasso (sketch)
For each feature $j$, minimize over $\theta_j$ holding others fixed:
$$\theta_j \leftarrow \text{soft\_threshold}\!\left(\frac{\rho_j}{z_j},\ \lambda\right)$$
where $\rho_j = X_j^T (y - X_{-j}\theta_{-j})$ is the partial residual and $z_j = \|X_j\|^2$.


In [ ]:
def soft_threshold(x, lam):
    """Lasso soft-thresholding / proximal operator."""
    return np.sign(x) * np.maximum(np.abs(x) - lam, 0)

def ridge_gd(X, y, lam=1.0, lr=0.01, n_iter=500):
    n, d = X.shape
    theta = np.zeros(d)
    for _ in range(n_iter):
        residuals = X @ theta - y
        grad = X.T @ residuals / n + 2 * lam * theta
        theta -= lr * grad
    return theta

def lasso_coordinate_descent(X, y, lam=0.1, n_iter=300):
    """Coordinate descent for Lasso."""
    n, d = X.shape
    theta = np.zeros(d)
    z = (X**2).sum(axis=0)  # precompute column norms squared
    for _ in range(n_iter):
        for j in range(d):
            residuals_j = y - X @ theta + X[:, j] * theta[j]  # partial residual
            rho_j = X[:, j] @ residuals_j
            theta[j] = soft_threshold(rho_j / z[j], lam / 2)
    return theta

from sklearn.linear_model import Ridge as SkRidge, Lasso as SkLasso

theta_ridge = ridge_gd(X_b, y, lam=1.0, lr=0.05, n_iter=500)
sk_ridge = SkRidge(alpha=1.0).fit(X, y)
sk_ridge_w = np.append(sk_ridge.coef_, sk_ridge.intercept_)

print("=== Ridge ===")
print(f"Scratch: {theta_ridge[:5].round(4)}")
print(f"sklearn: {sk_ridge_w[:5].round(4)}")

# Lasso (on non-bias features)
theta_lasso = lasso_coordinate_descent(X, y, lam=0.1, n_iter=500)
sk_lasso = SkLasso(alpha=0.1, max_iter=2000).fit(X, y)
print("\n=== Lasso ===")
print(f"Scratch (feat 3 should be ~0): {theta_lasso.round(4)}")
print(f"sklearn:                        {sk_lasso.coef_.round(4)}")


## Optimizer Zoo

| Optimizer | Update Rule | Hyperparams | When to use |
|---|---|---|---|
| **SGD** | $\theta \leftarrow \theta - \eta g$ | lr | Baseline; add momentum for speed |
| **SGD + Momentum** | $v \leftarrow \beta v + g$; $\theta \leftarrow \theta - \eta v$ | lr, β | Most standard DL training |
| **RMSProp** | $s \leftarrow \beta s + (1-\beta)g^2$; $\theta \leftarrow \theta - \eta g/\sqrt{s+\epsilon}$ | lr, β | RNNs, non-stationary |
| **Adam** | Combines momentum + RMSProp with bias correction | lr, β₁, β₂, ε | Default for most transformers |


In [ ]:
def make_optimizer(name, params):
    """Factory returning (init_state_fn, update_fn) pairs."""
    if name == 'sgd':
        lr = params.get('lr', 0.01)
        def init(theta): return {}
        def update(theta, grad, state, t):
            return theta - lr * grad, state
    elif name == 'momentum':
        lr, beta = params.get('lr', 0.01), params.get('beta', 0.9)
        def init(theta): return {'v': np.zeros_like(theta)}
        def update(theta, grad, state, t):
            state['v'] = beta * state['v'] + grad
            return theta - lr * state['v'], state
    elif name == 'adam':
        lr  = params.get('lr', 0.001)
        b1  = params.get('beta1', 0.9)
        b2  = params.get('beta2', 0.999)
        eps = params.get('eps', 1e-8)
        def init(theta): return {'m': np.zeros_like(theta), 'v': np.zeros_like(theta)}
        def update(theta, grad, state, t):
            state['m'] = b1 * state['m'] + (1 - b1) * grad
            state['v'] = b2 * state['v'] + (1 - b2) * grad**2
            m_hat = state['m'] / (1 - b1**t)   # bias correction
            v_hat = state['v'] / (1 - b2**t)
            return theta - lr * m_hat / (np.sqrt(v_hat) + eps), state
    return init, update

def train_with_optimizer(X, y, optimizer_name, opt_params, n_iter=300, batch_size=32):
    n, d = X.shape
    theta = np.zeros(d)
    init_fn, update_fn = make_optimizer(optimizer_name, opt_params)
    state = init_fn(theta)
    losses = []
    for t in range(1, n_iter + 1):
        idx = np.random.choice(n, batch_size, replace=False)
        Xb, yb = X[idx], y[idx]
        preds = Xb @ theta
        loss = 0.5 * np.mean((preds - yb)**2)
        grad = Xb.T @ (preds - yb) / batch_size
        theta, state = update_fn(theta, grad, state, t)
        losses.append(loss)
    return theta, losses

results = {}
for opt, params in [('sgd', {'lr': 0.05}),
                     ('momentum', {'lr': 0.05, 'beta': 0.9}),
                     ('adam', {'lr': 0.01})]:
    _, losses = train_with_optimizer(X_b, y, opt, params, n_iter=400)
    results[opt] = losses
    print(f"{opt:10s} final loss: {losses[-1]:.4f}")


## Learning-Rate Diagnostics

> 💡 **Interview Tip:** When asked "how do you tune the learning rate?", describe the loss-curve signatures: diverging (too high), very slow descent (too low), fast early then stuck (reasonable start, needs decay), smooth decay to low plateau (good). Then mention warmup + cosine decay for transformers.


In [ ]:
# Demonstrate too-high / too-low learning rates
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

for lr, label in [(0.5, 'too high (diverges)'),
                  (0.001, 'too low (slow)'),
                  (0.05, 'good')]:
    _, losses = train_with_optimizer(X_b, y, 'sgd', {'lr': lr}, n_iter=300)
    smoothed = np.convolve(losses, np.ones(10)/10, mode='valid')
    axes[0].plot(smoothed[:100], label=f'lr={lr} ({label})')

axes[0].set_title('Learning rate comparison (first 100 steps)')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=7)
axes[0].set_ylim(0, 5)

# Standardization necessity demo
X_skewed = X.copy()
X_skewed[:, 0] *= 100   # Feature 0 has 100x larger scale
X_skewed_b = np.hstack([X_skewed, np.ones((n, 1))])
y_skewed = X_skewed @ true_w + 0.3 * np.random.randn(n)

_, losses_unstd = train_with_optimizer(X_skewed_b, y_skewed, 'sgd', {'lr': 0.001}, n_iter=300)

scaler_X = X_skewed - X_skewed.mean(axis=0)
scaler_X /= (X_skewed.std(axis=0) + 1e-8)
scaler_X_b = np.hstack([scaler_X, np.ones((n, 1))])
_, losses_std = train_with_optimizer(scaler_X_b, y_skewed, 'sgd', {'lr': 0.05}, n_iter=300)

axes[1].plot(np.convolve(losses_unstd, np.ones(10)/10, mode='valid'), label='unstandarized (lr=0.001)')
axes[1].plot(np.convolve(losses_std,   np.ones(10)/10, mode='valid'), label='standardized (lr=0.05)')
axes[1].set_title('Standardization effect on convergence')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/lr_diagnostics.png', dpi=80)
plt.close()
print("Figure saved (would render inline in Jupyter)")
print(f"Unstandardized final loss: {losses_unstd[-1]:.4f}")
print(f"Standardized final loss:   {losses_std[-1]:.4f}")


## Common Interview Questions

**Q: Derive the gradient of MSE loss with respect to θ.**
$L = \frac{1}{2n}\|X\theta - y\|^2$. Expanding: $L = \frac{1}{2n}(X\theta-y)^T(X\theta-y)$. Taking the gradient: $\nabla_\theta L = \frac{1}{n}X^T(X\theta-y)$. The $\frac{1}{2}$ cancels the 2 from the squared norm derivative.

**Q: Why doesn't the normal equation work at large scale?**
Inverting $X^TX$ costs $O(d^3)$ and storing it costs $O(d^2)$. For $d = 10^6$ features this is infeasible. Gradient descent with mini-batches is $O(bd)$ per step, which scales to billions of parameters.

**Q: What is the difference between SGD and mini-batch GD?**
True SGD uses a single sample per step (very noisy). Mini-batch uses $b$ samples; this reduces gradient variance by $\approx \sqrt{b}$ while keeping per-step cost low. In practice "SGD" almost always means mini-batch SGD.

**Q: Why does Adam converge faster than SGD in practice?**
Adam maintains per-parameter learning rates (via the second moment estimate), which effectively normalizes by gradient magnitude. Parameters with consistently large gradients get smaller steps; sparse parameters get larger steps. The bias correction prevents large early updates when the moment estimates are small.

**Q: Why do we need feature standardization before gradient descent?**
Without standardization, features on different scales cause the loss landscape to be elongated — gradient descent bounces back and forth rather than converging smoothly. Standardization makes the landscape more spherical, allowing a single learning rate to work for all features.

**Q: When would you use coordinate descent instead of gradient descent?**
For $L_1$-regularized problems (Lasso, elastic net), $L_1$ is non-differentiable at zero. Coordinate descent handles this via the soft-threshold operator and is typically faster than subgradient methods for these problems.

## Key Takeaways
- Normal equation gives exact solution in $O(d^3)$; prefer GD for $d \gtrsim 10^4$
- Linear and logistic regression gradients have the same form: $X^T(\hat{y}-y)/n$
- Ridge adds $2\lambda\theta$ to the gradient; Lasso requires coordinate descent + soft-thresholding
- Adam = momentum + RMSProp + bias correction; default for transformers
- Feature standardization is *required* for GD to converge reliably at a single learning rate
- Diagnose LR from loss curves: diverging = too high, crawling = too low, smooth decay = good
- Mini-batch GD is the practical middle ground between noisy SGD and expensive full-batch GD